# Mammography — ROI preparation

Preparation of 256 × 256 lesion-centered ROI crops for CBIS-DDSM and INbreast, with patient-level split checks and dataset integrity controls.


In [ ]:
from pathlib import Path

SOURCE_PATH = None

OUT_DIR = '/kaggle/working/ROI_Crops_256_v1'
RESIZE_SIZE = 256
MARGIN_RATIO = 0.40
SEED = 42
VAL_FRAC = 0.15
TEST_FRAC = 0.15
OVERWRITE = True

FINAL_ZIP = '/kaggle/working/ROI_Crops_256_v1_Kaggle.zip'

print('Configuration chargée.')


In [ ]:
from pathlib import Path
import hashlib
import zipfile
import shutil
import pandas as pd

INPUT_DIR = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print('Contenu direct de /kaggle/input :')
if INPUT_DIR.exists():
    direct_items = sorted(INPUT_DIR.iterdir())
    if not direct_items:
        print('  /kaggle/input est vide.')
    for p in direct_items:
        kind = 'DIR ' if p.is_dir() else 'FILE'
        print(f'  [{kind}] {p}')
else:
    raise FileNotFoundError('/kaggle/input n’existe pas. Êtes-vous bien dans Kaggle ?')

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def score_candidate(path):
    full = str(path).lower()
    score = 0
    for token in ['cbis', 'ddsm', 'inbreast', 'in_breast', 'in-breast', 'inbraest', '448']:
        if token in full:
            score += 3
    for token in ['mass', 'mamm', 'roi', 'full', 'field', 'npz', 'mask']:
        if token in full:
            score += 1
    return score

def count_useful_files(folder, max_scan=200000):
    counts = {'npz': 0, 'png': 0, 'jpg': 0, 'jpeg': 0, 'zip': 0, 'csv': 0}
    n = 0
    for p in folder.rglob('*'):
        if not p.is_file():
            continue
        n += 1
        ext = p.suffix.lower().lstrip('.')
        if ext in counts:
            counts[ext] += 1
        if n >= max_scan:
            break
    counts['scanned_files'] = n
    return counts

def make_zip_from_dir(source_dir, zip_path):
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    base = str(zip_path).replace('.zip', '')
    print(f'Création du ZIP temporaire depuis le dossier : {source_dir}')
    shutil.make_archive(base, 'zip', root_dir=source_dir)
    return zip_path

all_zips = sorted(INPUT_DIR.rglob('*.zip'))
print(f'\nZIP détectés dans /kaggle/input : {len(all_zips)}')
for p in all_zips[:200]:
    print(' -', p)
if len(all_zips) > 200:
    print(f' ... {len(all_zips) - 200} ZIP supplémentaires non affichés')

all_dirs = sorted([p for p in INPUT_DIR.rglob('*') if p.is_dir()])
ranked_dirs = []
for d in all_dirs:
    s = score_candidate(d)
    if s > 0 or d.parent == INPUT_DIR:
        try:
            counts = count_useful_files(d)
        except Exception as e:
            counts = {'error': str(e), 'scanned_files': 0, 'npz': 0, 'png': 0, 'jpg': 0, 'jpeg': 0, 'zip': 0, 'csv': 0}
        useful = counts.get('npz', 0) + counts.get('png', 0) + counts.get('jpg', 0) + counts.get('jpeg', 0) + counts.get('zip', 0)
        ranked_dirs.append((s + useful, s, useful, d, counts))
ranked_dirs = sorted(ranked_dirs, key=lambda x: (-x[0], str(x[3])))

print('\nDossiers candidats visibles :')
for total_score, name_score, useful, d, counts in ranked_dirs[:30]:
    print(f' score={total_score:06d} useful={useful:06d} | {d} | counts={counts}')
if not ranked_dirs:
    print(' Aucun dossier candidat avec fichiers utiles détecté.')

manual = SOURCE_PATH is not None
selected_source = None
source_type = None

if SOURCE_PATH is not None:
    p = Path(str(SOURCE_PATH))
    if p.exists():
        selected_source = p
        source_type = 'zip' if p.is_file() and p.suffix.lower() == '.zip' else 'dir' if p.is_dir() else 'file'
    else:
        print('\n[AVERTISSEMENT] SOURCE_PATH manuel introuvable :')
        print(' ', p)
        print('Rappel : utilisez /kaggle/input/<dataset-slug>/..., pas /kaggle/input/datasets/<owner>/...')
        print('Recherche automatique dans /kaggle/input...')

if selected_source is None:
    if all_zips:
        ranked_zips = sorted([(score_candidate(p), p) for p in all_zips], key=lambda x: (-x[0], str(x[1])))
        print('\nCandidats ZIP classés :')
        for score, p in ranked_zips[:30]:
            print(f' score={score:02d} | {p}')
        selected_source = ranked_zips[0][1]
        source_type = 'zip'
    else:
        dirs_with_files = [x for x in ranked_dirs if x[2] > 0]
        if not dirs_with_files:
            raise FileNotFoundError(
                "Aucun .zip et aucun dossier contenant .npz/.png/.jpg n'est visible dans /kaggle/input.\n"
                "Dans Kaggle, cliquez sur Add Data et ajoutez le dataset.\n"
                "Ensuite utilisez le chemin monté du type /kaggle/input/<dataset-slug>/, pas /kaggle/input/datasets/<owner>/<dataset>."
            )
        selected_source = dirs_with_files[0][3]
        source_type = 'dir'
        print('\nAucun ZIP visible : sélection automatique du meilleur dossier source :')
        print(' ', selected_source)

if source_type == 'file' and selected_source.suffix.lower() != '.zip':
    raise ValueError(f'SOURCE_PATH pointe vers un fichier non ZIP : {selected_source}')

if source_type == 'dir':
    SOURCE_ZIP = make_zip_from_dir(selected_source, WORKING_DIR / 'temporary_source_from_kaggle_input.zip')
    SOURCE_ORIGINAL_PATH = selected_source
    SOURCE_KIND = 'directory_zipped_temporarily'
elif source_type == 'zip':
    SOURCE_ZIP = selected_source
    SOURCE_ORIGINAL_PATH = selected_source
    SOURCE_KIND = 'zip'
else:
    raise RuntimeError(f'Type de source inattendu : {source_type}')

SOURCE_ZIP = Path(SOURCE_ZIP)
if not SOURCE_ZIP.exists():
    raise FileNotFoundError(f'SOURCE_ZIP final introuvable : {SOURCE_ZIP}')

print('\nSource originale retenue :', SOURCE_ORIGINAL_PATH)
print('Type source :', SOURCE_KIND)
print('ZIP transmis au générateur :', SOURCE_ZIP)
print('Taille ZIP :', SOURCE_ZIP.stat().st_size, 'octets')

print('Calcul SHA256...')
source_sha256 = sha256_file(SOURCE_ZIP)
print('SHA256:', source_sha256)

print('Test intégrité ZIP...')
with zipfile.ZipFile(SOURCE_ZIP, 'r') as zf:
    bad = zf.testzip()
if bad is not None:
    raise RuntimeError(f'ZIP corrompu, premier fichier problématique : {bad}')
print('Intégrité ZIP : OK')

pd.DataFrame([{
    'source_kind': SOURCE_KIND,
    'source_original_path': str(SOURCE_ORIGINAL_PATH),
    'source_zip_used': str(SOURCE_ZIP),
    'sha256': source_sha256,
    'size_bytes': SOURCE_ZIP.stat().st_size,
}]).to_csv(WORKING_DIR / 'roi_crops_256_source_zip_audit.csv', index=False)

print('Audit écrit :', WORKING_DIR / 'roi_crops_256_source_zip_audit.csv')


In [ ]:
from pathlib import Path

SCRIPT_PATH = Path('/kaggle/working/generate_roi_crops_256.py')
SCRIPT_CODE = r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
generate_roi_crops_256.py

Generate ROI crops 256x256 for mammography mass segmentation from a full-field
448x448 package containing CBIS-DDSM and INbreast samples.

Scientific contract:
- Crop is computed on the 448x448 source image, then resized once to 256x256.
- Image resize: bilinear interpolation.
- Mask resize: nearest-neighbor interpolation, then binary thresholding.
- INbreast is kept as sealed external test set.
- CBIS-DDSM is split at patient level into train / validation / test.
- Every excluded or unmatched case is written to roi_crop_failures.csv.

Dependencies: numpy, pandas, Pillow, tqdm
Optional: none
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import os
import random
import re
import shutil
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm


# -----------------------------------------------------------------------------
# Global constants
# -----------------------------------------------------------------------------

GENERATE_NEGATIVE_CROPS = False
EXPECTED_SOURCE_SIZE = 448
DEFAULT_RESIZE_SIZE = 256
MANIFEST_COLUMNS = [
    "sample_id", "dataset", "source", "split", "patient_id", "case_id", "laterality",
    "oracle_crop_flag", "lesion_ratio_original", "lesion_ratio_crop",
    "bbox_w", "bbox_h", "crop_size_native", "crop_touches_border", "npz_path"
]

NPZ_REQUIRED_KEYS = [
    "image", "mask", "sample_id", "dataset", "source", "patient_id", "case_id",
    "laterality", "split", "oracle_crop_flag", "orig_bbox", "expanded_bbox",
    "margin_ratio", "crop_size_native", "resize_size"
]

MASK_TOKENS = [
    "mask", "masks", "seg", "segmentation", "label", "labels", "gt", "groundtruth",
    "ground_truth", "annotation", "annotations", "lesion_mask"
]
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
ZIP_EXT = ".zip"


# -----------------------------------------------------------------------------
# Data structures
# -----------------------------------------------------------------------------

@dataclass
class SourcePair:
    image_path: Optional[Path]
    mask_path: Optional[Path]
    npz_path: Optional[Path]
    source: str
    dataset: str
    patient_id: str
    case_id: str
    laterality: str
    metadata_rule: str


@dataclass
class Failure:
    source: str
    dataset: str
    patient_id: str
    case_id: str
    laterality: str
    reason: str
    detail: str


# -----------------------------------------------------------------------------
# Utility helpers
# -----------------------------------------------------------------------------

def set_determinism(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def safe_extract_zip(zip_path: Path, dest_dir: Path) -> None:
    """Extract a ZIP archive with a basic zip-slip guard."""
    dest_dir = dest_dir.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError(f"Corrupted ZIP entry in {zip_path}: {bad}")
        for member in zf.infolist():
            member_name = member.filename
            if not member_name or member_name.endswith("/"):
                continue
            target = (dest_dir / member_name).resolve()
            if not str(target).startswith(str(dest_dir)):
                raise RuntimeError(f"Unsafe ZIP path detected: {member_name}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member, "r") as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)


def recursive_extract_zip(source_zip: Path, extract_root: Path) -> List[Path]:
    """Extract source_zip and all nested ZIP files recursively."""
    extracted_zips: List[Path] = []
    safe_extract_zip(source_zip, extract_root)
    processed: set[Path] = set()

    while True:
        nested = [p for p in extract_root.rglob("*.zip") if p.resolve() not in processed]
        if not nested:
            break
        for z in nested:
            processed.add(z.resolve())
            extracted_zips.append(z)
            nested_dir = z.parent / f"__unzipped__{z.stem}"
            nested_dir.mkdir(parents=True, exist_ok=True)
            try:
                safe_extract_zip(z, nested_dir)
            except Exception as e:
                print(f"[WARN] Nested ZIP extraction failed for {z}: {e}", file=sys.stderr)
    return extracted_zips


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def stable_sample_id(dataset: str, patient_id: str, case_id: str, laterality: str, source: str) -> str:
    base = f"{dataset}|{patient_id}|{case_id}|{laterality}|{source}"
    digest = hashlib.sha1(base.encode("utf-8")).hexdigest()[:12]
    clean_case = re.sub(r"[^A-Za-z0-9_\-]+", "_", case_id).strip("_")[:80]
    clean_dataset = "inbreast" if dataset.lower().startswith("in") else "cbis_ddsm"
    return f"{clean_dataset}_{clean_case}_{digest}"


def normalize_stem_for_pairing(path: Path) -> str:
    """Normalize a filename stem so image and mask variants can be paired."""
    s = path.stem.lower()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[()\[\]{}]", "_", s)
    # Remove common mask/segmentation tokens as independent tokens or suffixes.
    for token in sorted(MASK_TOKENS, key=len, reverse=True):
        s = re.sub(rf"(^|[_\-\.]){re.escape(token)}($|[_\-\.])", "_", s)
        s = re.sub(rf"[_\-\.]?{re.escape(token)}$", "", s)
    s = re.sub(r"[_\-.]+", "_", s).strip("_")
    return s


def is_mask_like_path(path: Path) -> bool:
    parts = [p.lower() for p in path.parts]
    stem = path.stem.lower()
    text = "/".join(parts)
    for token in MASK_TOKENS:
        if re.search(rf"(^|[\W_]){re.escape(token)}($|[\W_])", text):
            return True
        if stem.endswith("_" + token) or stem.startswith(token + "_"):
            return True
    return False


def infer_dataset(path_or_text: str) -> str:
    txt = path_or_text.lower()
    if "inbreast" in txt or "in_breast" in txt or "in-breast" in txt:
        return "INbreast"
    if "cbis" in txt or "ddsm" in txt or "mass-training" in txt or "mass-test" in txt:
        return "CBIS-DDSM"
    # Conservative default: CBIS-DDSM, because INbreast must not leak into train.
    return "CBIS-DDSM"


def infer_laterality(path_or_text: str) -> str:
    txt = path_or_text.upper()
    if re.search(r"(^|[^A-Z])LEFT([^A-Z]|$)", txt) or re.search(r"(^|[^A-Z])L([^A-Z]|$)", txt):
        return "LEFT"
    if re.search(r"(^|[^A-Z])RIGHT([^A-Z]|$)", txt) or re.search(r"(^|[^A-Z])R([^A-Z]|$)", txt):
        return "RIGHT"
    return "UNK"


def infer_patient_id(path_or_text: str, dataset: str, case_id: str) -> Tuple[str, str]:
    txt = str(path_or_text)
    upper = txt.upper()

    # CBIS common pattern: P_00001
    m = re.search(r"P[_\-]?\d{3,6}", upper)
    if m:
        return m.group(0).replace("-", "_"), "regex:P_00001"

    # Patient-specific directory/file naming.
    m = re.search(r"(?:PATIENT|PAT|PID|SUBJECT|SUBJ)[_\-\s]*(\d{2,8})", upper)
    if m:
        return f"PATIENT_{m.group(1)}", "regex:patient_numeric"

    # INbreast images are often numeric mammogram identifiers; use first long numeric token.
    if dataset == "INbreast":
        nums = re.findall(r"\d{4,8}", upper)
        if nums:
            return f"INBREAST_{nums[0]}", "regex:inbreast_numeric"

    # Fallback: use the case stem before laterality/view tokens; deterministic but conservative.
    c = case_id.upper()
    c = re.sub(r"(LEFT|RIGHT|_L_|_R_|\bL\b|\bR\b|CC|MLO)", "_", c)
    c = re.sub(r"[_\-\.]+", "_", c).strip("_")
    fallback = c.split("_")[0] if c else hashlib.sha1(txt.encode("utf-8")).hexdigest()[:10]
    return f"UNKNOWNPAT_{fallback}", "fallback:case_prefix"


def infer_case_id(path: Path) -> str:
    rel = str(path.with_suffix(""))
    rel = re.sub(r"__unzipped__[^/\\]+[/\\]", "", rel)
    rel = re.sub(r"[\s()\[\]{}]+", "_", rel)
    rel = re.sub(r"[^A-Za-z0-9_\-./\\]+", "_", rel)
    return normalize_stem_for_pairing(Path(rel)) or path.stem


def load_manifest_metadata(root: Path) -> Dict[str, Dict[str, str]]:
    """
    Load CSV manifests if present and build a forgiving lookup by basename/stem/path.
    This is intentionally permissive; filename parsing remains the fallback.
    """
    lookups: Dict[str, Dict[str, str]] = {}
    csv_files = [p for p in root.rglob("*.csv") if p.is_file()]
    for csv_path in csv_files:
        try:
            df = pd.read_csv(csv_path, dtype=str, low_memory=False)
        except Exception:
            continue
        lower_cols = {c.lower(): c for c in df.columns}
        meta_cols = {}
        for key in ["dataset", "patient_id", "patient", "case_id", "case", "laterality", "side", "split", "source"]:
            if key in lower_cols:
                meta_cols[key] = lower_cols[key]

        path_cols = [
            c for c in df.columns
            if any(tok in c.lower() for tok in ["path", "file", "filename", "npz", "image", "mask", "sample_id", "case_id"])
        ]
        if not path_cols:
            continue

        for _, row in df.iterrows():
            meta = {}
            if "dataset" in meta_cols:
                meta["dataset"] = str(row[meta_cols["dataset"]])
            if "patient_id" in meta_cols:
                meta["patient_id"] = str(row[meta_cols["patient_id"]])
            elif "patient" in meta_cols:
                meta["patient_id"] = str(row[meta_cols["patient"]])
            if "case_id" in meta_cols:
                meta["case_id"] = str(row[meta_cols["case_id"]])
            elif "case" in meta_cols:
                meta["case_id"] = str(row[meta_cols["case"]])
            if "laterality" in meta_cols:
                meta["laterality"] = str(row[meta_cols["laterality"]])
            elif "side" in meta_cols:
                meta["laterality"] = str(row[meta_cols["side"]])
            if "source" in meta_cols:
                meta["source"] = str(row[meta_cols["source"]])

            if not meta:
                continue
            for pc in path_cols:
                value = row.get(pc)
                if value is None or (isinstance(value, float) and math.isnan(value)):
                    continue
                s = str(value)
                if not s or s.lower() == "nan":
                    continue
                keys = {s, Path(s).name, Path(s).stem, normalize_stem_for_pairing(Path(s))}
                for k in keys:
                    if k:
                        lookups[k.lower()] = meta
    return lookups


def apply_manifest_or_parse(path: Path, manifest_lookup: Dict[str, Dict[str, str]]) -> Tuple[str, str, str, str, str]:
    candidates = [str(path), path.name, path.stem, normalize_stem_for_pairing(path)]
    meta: Dict[str, str] = {}
    for c in candidates:
        if c and c.lower() in manifest_lookup:
            meta = manifest_lookup[c.lower()]
            break

    dataset = meta.get("dataset") if meta.get("dataset") else infer_dataset(str(path))
    dataset = "INbreast" if "in" in dataset.lower() and "breast" in dataset.lower() else dataset
    if dataset not in {"CBIS-DDSM", "INbreast"}:
        dataset = infer_dataset(str(path))

    case_id = meta.get("case_id") if meta.get("case_id") else infer_case_id(path)
    laterality = meta.get("laterality") if meta.get("laterality") else infer_laterality(str(path))
    laterality = str(laterality).upper()
    if laterality in {"L", "LEFT BREAST"}:
        laterality = "LEFT"
    elif laterality in {"R", "RIGHT BREAST"}:
        laterality = "RIGHT"
    elif laterality not in {"LEFT", "RIGHT"}:
        laterality = infer_laterality(str(path))

    if meta.get("patient_id"):
        patient_id = str(meta["patient_id"])
        rule = "manifest"
    else:
        patient_id, rule = infer_patient_id(str(path), dataset, case_id)
    return dataset, patient_id, case_id, laterality, rule


# -----------------------------------------------------------------------------
# Discovery and loading
# -----------------------------------------------------------------------------

def load_npz_pair(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    with np.load(path, allow_pickle=True) as data:
        if "image" not in data.files or "mask" not in data.files:
            raise KeyError(f"NPZ does not contain both 'image' and 'mask'. Keys={data.files}")
        image = np.asarray(data["image"])
        mask = np.asarray(data["mask"])
    return canonicalize_image(image), canonicalize_mask(mask)


def load_png(path: Path, is_mask: bool) -> np.ndarray:
    img = Image.open(path).convert("L")
    arr = np.asarray(img)
    if is_mask:
        return (arr > 0).astype(np.uint8)
    arr = arr.astype(np.float32)
    if arr.max() > 1.0:
        arr /= 255.0
    return np.clip(arr, 0.0, 1.0).astype(np.float32)


def canonicalize_image(image: np.ndarray) -> np.ndarray:
    image = np.asarray(image)
    image = np.squeeze(image)
    if image.ndim != 2:
        raise ValueError(f"Image must be 2D after squeeze, got shape {image.shape}")
    image = image.astype(np.float32)
    if not np.isfinite(image).all():
        raise ValueError("Image contains NaN or inf")
    # If image is not already in [0,1], robustly normalize min-max as a last-resort guard.
    mn, mx = float(image.min()), float(image.max())
    if mn < -1e-4 or mx > 1.0001:
        if mx > mn:
            image = (image - mn) / (mx - mn)
        else:
            image = np.zeros_like(image, dtype=np.float32)
    return np.clip(image, 0.0, 1.0).astype(np.float32)


def canonicalize_mask(mask: np.ndarray) -> np.ndarray:
    mask = np.asarray(mask)
    mask = np.squeeze(mask)
    if mask.ndim != 2:
        raise ValueError(f"Mask must be 2D after squeeze, got shape {mask.shape}")
    return (mask > 0).astype(np.uint8)


def discover_pairs(root: Path, failures: List[Failure]) -> List[SourcePair]:
    manifest_lookup = load_manifest_metadata(root)
    pairs: List[SourcePair] = []

    # 1) Self-contained NPZ files with image+mask.
    npz_files = sorted([p for p in root.rglob("*.npz") if p.is_file()])
    for p in npz_files:
        try:
            with np.load(p, allow_pickle=True) as data:
                has_pair = "image" in data.files and "mask" in data.files
        except Exception as e:
            dataset, patient_id, case_id, laterality, _ = apply_manifest_or_parse(p, manifest_lookup)
            failures.append(Failure(str(p), dataset, patient_id, case_id, laterality, "npz_unreadable", str(e)))
            continue
        if has_pair:
            dataset, patient_id, case_id, laterality, rule = apply_manifest_or_parse(p, manifest_lookup)
            pairs.append(SourcePair(
                image_path=None, mask_path=None, npz_path=p, source=str(p.relative_to(root)),
                dataset=dataset, patient_id=patient_id, case_id=case_id,
                laterality=laterality, metadata_rule=rule,
            ))

    # 2) PNG/TIFF/JPG image-mask pairs if present.
    image_files = sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    masks_by_id: Dict[str, List[Path]] = defaultdict(list)
    images_by_id: Dict[str, List[Path]] = defaultdict(list)
    for p in image_files:
        key = normalize_stem_for_pairing(p)
        if is_mask_like_path(p):
            masks_by_id[key].append(p)
        else:
            images_by_id[key].append(p)

    used_masks: set[Path] = set()
    for key, img_list in images_by_id.items():
        mask_list = masks_by_id.get(key, [])
        if not mask_list:
            for img in img_list:
                dataset, patient_id, case_id, laterality, _ = apply_manifest_or_parse(img, manifest_lookup)
                failures.append(Failure(str(img.relative_to(root)), dataset, patient_id, case_id, laterality, "image_without_mask", key))
            continue
        # Pair deterministically. If multiple, pair by sorted order and log extras.
        for img, msk in zip(sorted(img_list), sorted(mask_list)):
            used_masks.add(msk)
            dataset, patient_id, case_id, laterality, rule = apply_manifest_or_parse(img, manifest_lookup)
            pairs.append(SourcePair(
                image_path=img, mask_path=msk, npz_path=None, source=str(img.relative_to(root)),
                dataset=dataset, patient_id=patient_id, case_id=case_id,
                laterality=laterality, metadata_rule=rule,
            ))
        if len(img_list) != len(mask_list):
            longer, reason = (img_list, "extra_images_for_mask_key") if len(img_list) > len(mask_list) else (mask_list, "extra_masks_for_image_key")
            for extra in sorted(longer[min(len(img_list), len(mask_list)):]):
                dataset, patient_id, case_id, laterality, _ = apply_manifest_or_parse(extra, manifest_lookup)
                failures.append(Failure(str(extra.relative_to(root)), dataset, patient_id, case_id, laterality, reason, key))

    for key, mask_list in masks_by_id.items():
        for msk in mask_list:
            if msk not in used_masks and key not in images_by_id:
                dataset, patient_id, case_id, laterality, _ = apply_manifest_or_parse(msk, manifest_lookup)
                failures.append(Failure(str(msk.relative_to(root)), dataset, patient_id, case_id, laterality, "orphan_mask", key))

    # Remove exact duplicate source entries, preserving order.
    seen = set()
    unique_pairs = []
    for pair in pairs:
        key = (str(pair.npz_path), str(pair.image_path), str(pair.mask_path))
        if key not in seen:
            seen.add(key)
            unique_pairs.append(pair)
    return unique_pairs


def load_pair(pair: SourcePair) -> Tuple[np.ndarray, np.ndarray]:
    if pair.npz_path is not None:
        return load_npz_pair(pair.npz_path)
    if pair.image_path is None or pair.mask_path is None:
        raise RuntimeError("Invalid SourcePair: no NPZ and missing PNG image/mask paths")
    return load_png(pair.image_path, is_mask=False), load_png(pair.mask_path, is_mask=True)


# -----------------------------------------------------------------------------
# Split logic
# -----------------------------------------------------------------------------

def patient_level_split_cbis(pairs: Sequence[SourcePair], seed: int, val_frac: float, test_frac: float) -> Dict[str, str]:
    """Return mapping patient_id -> split for CBIS-DDSM only."""
    patient_lats: Dict[str, List[str]] = defaultdict(list)
    for p in pairs:
        patient_lats[p.patient_id].append(p.laterality)

    patients = sorted(patient_lats.keys())
    n = len(patients)
    if n < 3:
        raise RuntimeError(
            f"CBIS patient-level split needs at least 3 unique patients to create train/validation/test; got {n}."
        )

    def split_counts(num: int) -> Tuple[int, int, int]:
        n_test = max(1, int(round(num * test_frac)))
        n_val = max(1, int(round(num * val_frac)))
        if n_test + n_val >= num:
            n_test = 1
            n_val = 1
        n_train = num - n_val - n_test
        return n_train, n_val, n_test

    rng = random.Random(seed)
    labels = {}
    for pat, lats in patient_lats.items():
        labels[pat] = Counter(lats).most_common(1)[0][0] if lats else "UNK"
    label_counts = Counter(labels.values())
    can_stratify = len(label_counts) > 1 and all(c >= 3 for c in label_counts.values())

    mapping: Dict[str, str] = {}
    if can_stratify:
        for lab in sorted(label_counts):
            group = [p for p in patients if labels[p] == lab]
            rng.shuffle(group)
            _, n_val, n_test = split_counts(len(group))
            test_group = group[:n_test]
            val_group = group[n_test:n_test + n_val]
            train_group = group[n_test + n_val:]
            mapping.update({p: "test" for p in test_group})
            mapping.update({p: "validation" for p in val_group})
            mapping.update({p: "train" for p in train_group})
    else:
        group = patients[:]
        rng.shuffle(group)
        _, n_val, n_test = split_counts(len(group))
        test_group = group[:n_test]
        val_group = group[n_test:n_test + n_val]
        train_group = group[n_test + n_val:]
        mapping.update({p: "test" for p in test_group})
        mapping.update({p: "validation" for p in val_group})
        mapping.update({p: "train" for p in train_group})

    # Guard: all splits must exist.
    split_set = set(mapping.values())
    if not {"train", "validation", "test"}.issubset(split_set):
        raise RuntimeError(f"CBIS split failed to create all buckets. Splits={split_set}")
    return mapping


def assign_splits(pairs: Sequence[SourcePair], seed: int, val_frac: float, test_frac: float) -> Dict[int, str]:
    cbis_pairs = [p for p in pairs if p.dataset == "CBIS-DDSM"]
    cbis_mapping = patient_level_split_cbis(cbis_pairs, seed, val_frac, test_frac) if cbis_pairs else {}
    split_by_idx: Dict[int, str] = {}
    for i, pair in enumerate(pairs):
        if pair.dataset == "INbreast":
            split_by_idx[i] = "external_inbreast"
        elif pair.dataset == "CBIS-DDSM":
            split_by_idx[i] = cbis_mapping[pair.patient_id]
        else:
            raise RuntimeError(f"Unknown dataset for pair {pair.source}: {pair.dataset}")
    return split_by_idx


def assert_no_patient_leakage(pairs: Sequence[SourcePair], split_by_idx: Dict[int, str]) -> None:
    by_patient_dataset: Dict[Tuple[str, str], set] = defaultdict(set)
    for i, pair in enumerate(pairs):
        by_patient_dataset[(pair.dataset, pair.patient_id)].add(split_by_idx[i])
    leaks = []
    for (dataset, patient), splits in by_patient_dataset.items():
        if len(splits) > 1:
            leaks.append({"dataset": dataset, "patient_id": patient, "splits": sorted(splits)})
    if leaks:
        raise RuntimeError("Patient-level leakage detected: " + json.dumps(leaks[:20], ensure_ascii=False))


# -----------------------------------------------------------------------------
# Crop logic
# -----------------------------------------------------------------------------

def compute_bbox(mask: np.ndarray) -> Tuple[int, int, int, int]:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        raise ValueError("empty_mask")
    x0, x1 = int(xs.min()), int(xs.max()) + 1
    y0, y1 = int(ys.min()), int(ys.max()) + 1
    return x0, y0, x1, y1


def expanded_square_bbox(
    bbox: Tuple[int, int, int, int],
    image_shape: Tuple[int, int],
    margin_ratio: float,
) -> Tuple[Tuple[int, int, int, int], int, bool]:
    """
    Expand bbox by margin_ratio in x/y, then make a square centered on lesion.
    Returns half-open coordinates that may lie outside the image.
    """
    x0, y0, x1, y1 = bbox
    h_img, w_img = image_shape
    w = x1 - x0
    h = y1 - y0
    if w <= 0 or h <= 0:
        raise ValueError(f"degenerate_bbox:{bbox}")

    ex0 = x0 - margin_ratio * w
    ex1 = x1 + margin_ratio * w
    ey0 = y0 - margin_ratio * h
    ey1 = y1 + margin_ratio * h

    cx = (ex0 + ex1) / 2.0
    cy = (ey0 + ey1) / 2.0
    side = int(math.ceil(max(ex1 - ex0, ey1 - ey0)))
    side = max(1, side)

    sx0 = int(math.floor(cx - side / 2.0))
    sy0 = int(math.floor(cy - side / 2.0))
    sx1 = sx0 + side
    sy1 = sy0 + side
    touches = sx0 < 0 or sy0 < 0 or sx1 > w_img or sy1 > h_img
    return (sx0, sy0, sx1, sy1), side, touches


def crop_with_zero_padding(arr: np.ndarray, square: Tuple[int, int, int, int], fill_value: float = 0.0) -> np.ndarray:
    x0, y0, x1, y1 = square
    side = x1 - x0
    if side <= 0 or (y1 - y0) != side:
        raise ValueError(f"Invalid square crop coordinates: {square}")
    out = np.full((side, side), fill_value, dtype=arr.dtype)
    h, w = arr.shape

    src_x0 = max(0, x0)
    src_y0 = max(0, y0)
    src_x1 = min(w, x1)
    src_y1 = min(h, y1)
    if src_x1 <= src_x0 or src_y1 <= src_y0:
        raise ValueError(f"Crop does not intersect image: crop={square}, image_shape={arr.shape}")

    dst_x0 = src_x0 - x0
    dst_y0 = src_y0 - y0
    out[dst_y0:dst_y0 + (src_y1 - src_y0), dst_x0:dst_x0 + (src_x1 - src_x0)] = arr[src_y0:src_y1, src_x0:src_x1]
    return out


def resize_image_and_mask(image_crop: np.ndarray, mask_crop: np.ndarray, resize_size: int) -> Tuple[np.ndarray, np.ndarray]:
    image_pil = Image.fromarray(image_crop.astype(np.float32), mode="F")
    image_resized = image_pil.resize((resize_size, resize_size), resample=Image.BILINEAR)
    image_arr = np.asarray(image_resized, dtype=np.float32)
    image_arr = np.clip(image_arr, 0.0, 1.0).astype(np.float32)

    mask_pil = Image.fromarray((mask_crop > 0).astype(np.uint8) * 255, mode="L")
    mask_resized = mask_pil.resize((resize_size, resize_size), resample=Image.NEAREST)
    mask_arr = (np.asarray(mask_resized) >= 128).astype(np.uint8)
    return image_arr, mask_arr


def mask_contour(mask: np.ndarray) -> np.ndarray:
    m = mask.astype(bool)
    if not m.any():
        return np.zeros_like(mask, dtype=bool)
    p = np.pad(m, 1, mode="constant", constant_values=False)
    neighbors_all = (
        p[1:-1, :-2] & p[1:-1, 2:] & p[:-2, 1:-1] & p[2:, 1:-1]
    )
    return m & (~neighbors_all)


def save_overlay_png(image: np.ndarray, mask: np.ndarray, out_path: Path) -> None:
    gray = (np.clip(image, 0.0, 1.0) * 255).astype(np.uint8)
    rgb = np.stack([gray, gray, gray], axis=-1)
    contour = mask_contour(mask)
    rgb[contour] = np.array([255, 0, 0], dtype=np.uint8)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(rgb).save(out_path)


# -----------------------------------------------------------------------------
# Output validation and summaries
# -----------------------------------------------------------------------------

def write_npz(
    out_path: Path,
    image: np.ndarray,
    mask: np.ndarray,
    sample_id: str,
    pair: SourcePair,
    split: str,
    oracle_crop_flag: bool,
    orig_bbox: Tuple[int, int, int, int],
    expanded_bbox: Tuple[int, int, int, int],
    margin_ratio: float,
    crop_size_native: int,
    resize_size: int,
) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        out_path,
        image=image.astype(np.float32),
        mask=mask.astype(np.uint8),
        sample_id=np.array(sample_id),
        dataset=np.array(pair.dataset),
        source=np.array(pair.source),
        patient_id=np.array(pair.patient_id),
        case_id=np.array(pair.case_id),
        laterality=np.array(pair.laterality),
        split=np.array(split),
        oracle_crop_flag=np.array(bool(oracle_crop_flag)),
        orig_bbox=np.array(orig_bbox, dtype=np.int32),
        expanded_bbox=np.array(expanded_bbox, dtype=np.int32),
        margin_ratio=np.array(float(margin_ratio), dtype=np.float32),
        crop_size_native=np.array(int(crop_size_native), dtype=np.int32),
        resize_size=np.array(int(resize_size), dtype=np.int32),
    )


def validate_npz_file(path: Path, root: Path, resize_size: int) -> Dict[str, object]:
    row: Dict[str, object] = {
        "npz_path": str(path.relative_to(root)),
        "ok": False,
        "error": "",
    }
    try:
        with np.load(path, allow_pickle=True) as data:
            keys = set(data.files)
            missing = [k for k in NPZ_REQUIRED_KEYS if k not in keys]
            row["keys_present"] = ";".join(sorted(keys))
            row["missing_keys"] = ";".join(missing)
            image = np.asarray(data["image"]) if "image" in keys else None
            mask = np.asarray(data["mask"]) if "mask" in keys else None
            if missing:
                raise ValueError(f"missing_keys:{missing}")
            if image.shape != (resize_size, resize_size):
                raise ValueError(f"bad_image_shape:{image.shape}")
            if mask.shape != (resize_size, resize_size):
                raise ValueError(f"bad_mask_shape:{mask.shape}")
            if image.dtype != np.float32:
                raise ValueError(f"bad_image_dtype:{image.dtype}")
            if mask.dtype != np.uint8:
                raise ValueError(f"bad_mask_dtype:{mask.dtype}")
            if not np.isfinite(image).all():
                raise ValueError("image_non_finite")
            if float(image.min()) < -1e-6 or float(image.max()) > 1.0 + 1e-6:
                raise ValueError(f"image_out_of_range:min={image.min()},max={image.max()}")
            uniq = set(np.unique(mask).tolist())
            if not uniq.issubset({0, 1}):
                raise ValueError(f"mask_not_binary:{sorted(uniq)}")
            if int(mask.sum()) == 0:
                raise ValueError("mask_empty")
            row.update({
                "image_shape": str(tuple(image.shape)),
                "mask_shape": str(tuple(mask.shape)),
                "image_dtype": str(image.dtype),
                "mask_dtype": str(mask.dtype),
                "image_min": float(image.min()),
                "image_max": float(image.max()),
                "mask_sum": int(mask.sum()),
                "ok": True,
            })
    except Exception as e:
        row["error"] = str(e)
    return row


def create_contact_sheet(overlay_paths: Sequence[Path], out_path: Path, max_images: int = 64, tile_size: int = 128) -> None:
    if not overlay_paths:
        return
    selected = list(overlay_paths)[:max_images]
    n = len(selected)
    cols = min(8, max(1, int(math.ceil(math.sqrt(n)))))
    rows = int(math.ceil(n / cols))
    canvas = Image.new("RGB", (cols * tile_size, rows * tile_size), color=(255, 255, 255))
    for idx, p in enumerate(selected):
        try:
            im = Image.open(p).convert("RGB").resize((tile_size, tile_size), Image.BILINEAR)
        except Exception:
            continue
        x = (idx % cols) * tile_size
        y = (idx // cols) * tile_size
        canvas.paste(im, (x, y))
    out_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(out_path)


def write_dataset_card(out_root: Path, config: Dict[str, object]) -> None:
    text = f"""# ROI Crops 256 Dataset Card

## Purpose
This dataset contains 256x256 ROI crops for binary mammographic mass segmentation.
Crops are generated from full-field 448x448 image/mask pairs. The crop is computed on
the 448x448 source image and resized once to 256x256.

## Experimental role
- CBIS-DDSM: patient-level train / validation / test split.
- INbreast: sealed external test bucket only (`external_inbreast`).
- The CBIS test bucket is intentionally present to correct the previous ROI protocol.

## Representation
Each sample is stored as a compressed NPZ with:
- `image`: 256x256 float32 in [0,1], single native grayscale channel.
- `mask`: 256x256 uint8 binary {{0,1}}.
- Metadata: sample_id, dataset, source, patient_id, case_id, laterality, split,
  oracle_crop_flag, bounding boxes, margin ratio, native crop size and resize size.

## Cropping policy
- Bounding box is computed from the positive reference mask.
- Margin ratio: {config.get('margin_ratio')}.
- The expanded crop is made square and zero-padded if it crosses image borders.
- Image interpolation: bilinear.
- Mask interpolation: nearest-neighbor, then binary re-thresholding.

## Negative crops
`GENERATE_NEGATIVE_CROPS = False`.
This dataset is therefore a localization-aware segmentation setting. It assumes that a
lesion location is known or proposed. It should not be presented as full-field detection.

## Governance and limitations
- No case is silently discarded; exclusions are listed in `roi_crop_failures.csv`.
- Patient-level leakage is checked programmatically.
- INbreast is never mixed with CBIS train/validation/test.
- Since crops are derived from ground-truth masks, validation/test/external crops are
  oracle crops and must be described as such in the manuscript.
"""
    (out_root / "ROI_CROPS_DATASET_CARD.md").write_text(text, encoding="utf-8")


def write_failures_csv(failures: Sequence[Failure], out_path: Path) -> None:
    rows = [asdict(f) for f in failures]
    if rows:
        pd.DataFrame(rows).to_csv(out_path, index=False)
    else:
        pd.DataFrame(columns=["source", "dataset", "patient_id", "case_id", "laterality", "reason", "detail"]).to_csv(out_path, index=False)


# -----------------------------------------------------------------------------
# Main pipeline
# -----------------------------------------------------------------------------

def generate_dataset(args: argparse.Namespace) -> None:
    set_determinism(args.seed)
    source_zip = Path(args.source_zip)
    out_root = Path(args.out_dir)
    if not source_zip.exists():
        raise FileNotFoundError(f"Source ZIP not found: {source_zip}")
    if source_zip.suffix.lower() != ZIP_EXT:
        raise ValueError(f"--source-zip must point to a .zip file, got {source_zip}")

    if out_root.exists() and args.overwrite:
        shutil.rmtree(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    for split in ["train", "validation", "test", "external_inbreast"]:
        (out_root / split).mkdir(parents=True, exist_ok=True)

    failures: List[Failure] = []
    manifest_rows: List[Dict[str, object]] = []
    overlay_paths: List[Path] = []

    config = {
        "source_zip": str(source_zip),
        "source_zip_sha256": sha256_file(source_zip),
        "out_dir": str(out_root),
        "resize_size": int(args.resize_size),
        "expected_source_size": EXPECTED_SOURCE_SIZE,
        "margin_ratio": float(args.margin_ratio),
        "seed": int(args.seed),
        "val_frac": float(args.val_frac),
        "test_frac": float(args.test_frac),
        "generate_negative_crops": GENERATE_NEGATIVE_CROPS,
        "image_interpolation": "bilinear",
        "mask_interpolation": "nearest_neighbor",
        "mask_threshold_after_resize": 0.5,
        "split_policy": "CBIS-DDSM patient-level 70/15/15; INbreast external_inbreast only",
    }
    (out_root / "roi_crop_config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")

    with tempfile.TemporaryDirectory(prefix="roi_crops_448_extract_") as tmp:
        extract_root = Path(tmp) / "extracted"
        extract_root.mkdir(parents=True, exist_ok=True)
        print(f"[1/7] Extracting source ZIP recursively: {source_zip}")
        nested_zips = recursive_extract_zip(source_zip, extract_root)
        print(f"      Nested ZIP files extracted: {len(nested_zips)}")

        print("[2/7] Discovering NPZ and PNG image/mask pairs")
        pairs = discover_pairs(extract_root, failures)
        print(f"      Candidate paired samples: {len(pairs)}")
        if not pairs:
            write_failures_csv(failures, out_root / "roi_crop_failures.csv")
            raise RuntimeError("No valid image/mask pairs discovered. See roi_crop_failures.csv.")

        counts_dataset = Counter(p.dataset for p in pairs)
        print(f"      Dataset counts before filtering: {dict(counts_dataset)}")

        print("[3/7] Assigning patient-level splits")
        split_by_idx = assign_splits(pairs, args.seed, args.val_frac, args.test_frac)
        assert_no_patient_leakage(pairs, split_by_idx)
        print("      Patient leakage check: OK")

        print("[4/7] Generating ROI crops")
        for i, pair in enumerate(tqdm(pairs, desc="crops", unit="sample")):
            split = split_by_idx[i]
            try:
                image, mask = load_pair(pair)
                image = canonicalize_image(image)
                mask = canonicalize_mask(mask)

                if image.shape != mask.shape:
                    raise ValueError(f"image_mask_shape_mismatch:image={image.shape},mask={mask.shape}")
                if image.shape != (EXPECTED_SOURCE_SIZE, EXPECTED_SOURCE_SIZE):
                    raise ValueError(f"source_not_448x448:shape={image.shape}")
                if int(mask.sum()) == 0:
                    if GENERATE_NEGATIVE_CROPS:
                        raise NotImplementedError("Negative crop generation is intentionally disabled.")
                    raise ValueError("empty_mask")

                orig_bbox = compute_bbox(mask)
                x0, y0, x1, y1 = orig_bbox
                bbox_w, bbox_h = x1 - x0, y1 - y0
                expanded_bbox, crop_size_native, touches_border = expanded_square_bbox(
                    orig_bbox, image.shape, args.margin_ratio
                )
                image_crop_native = crop_with_zero_padding(image, expanded_bbox, fill_value=0.0)
                mask_crop_native = crop_with_zero_padding(mask, expanded_bbox, fill_value=0)
                image_256, mask_256 = resize_image_and_mask(image_crop_native, mask_crop_native, args.resize_size)

                if int(mask_256.sum()) == 0:
                    raise ValueError("mask_empty_after_resize")
                if image_256.shape != (args.resize_size, args.resize_size):
                    raise ValueError(f"bad_resized_image_shape:{image_256.shape}")
                if mask_256.shape != (args.resize_size, args.resize_size):
                    raise ValueError(f"bad_resized_mask_shape:{mask_256.shape}")

                oracle_crop_flag = split != "train"
                sample_id = stable_sample_id(pair.dataset, pair.patient_id, pair.case_id, pair.laterality, pair.source)
                npz_rel = Path(split) / f"{sample_id}.npz"
                overlay_rel = Path(split) / f"{sample_id}_overlay.png"
                write_npz(
                    out_root / npz_rel,
                    image_256,
                    mask_256,
                    sample_id,
                    pair,
                    split,
                    oracle_crop_flag,
                    orig_bbox,
                    expanded_bbox,
                    args.margin_ratio,
                    crop_size_native,
                    args.resize_size,
                )
                save_overlay_png(image_256, mask_256, out_root / overlay_rel)
                overlay_paths.append(out_root / overlay_rel)

                lesion_ratio_original = float(mask.sum()) / float(mask.size)
                lesion_ratio_crop = float(mask_256.sum()) / float(mask_256.size)
                manifest_rows.append({
                    "sample_id": sample_id,
                    "dataset": pair.dataset,
                    "source": pair.source,
                    "split": split,
                    "patient_id": pair.patient_id,
                    "case_id": pair.case_id,
                    "laterality": pair.laterality,
                    "oracle_crop_flag": bool(oracle_crop_flag),
                    "lesion_ratio_original": lesion_ratio_original,
                    "lesion_ratio_crop": lesion_ratio_crop,
                    "bbox_w": int(bbox_w),
                    "bbox_h": int(bbox_h),
                    "crop_size_native": int(crop_size_native),
                    "crop_touches_border": bool(touches_border),
                    "npz_path": str(npz_rel),
                })
            except Exception as e:
                failures.append(Failure(
                    source=pair.source,
                    dataset=pair.dataset,
                    patient_id=pair.patient_id,
                    case_id=pair.case_id,
                    laterality=pair.laterality,
                    reason="crop_generation_failed",
                    detail=str(e),
                ))

    print("[5/7] Writing manifests and QC files")
    manifest_df = pd.DataFrame(manifest_rows, columns=MANIFEST_COLUMNS)
    manifest_df.to_csv(out_root / "roi_crop_manifest.csv", index=False)
    write_failures_csv(failures, out_root / "roi_crop_failures.csv")

    if manifest_df.empty:
        raise RuntimeError("No crops were generated. See roi_crop_failures.csv.")

    # Summary by dataset/split plus enrichment stats.
    summary = (
        manifest_df.assign(
            enrichment=lambda d: d["lesion_ratio_crop"] / d["lesion_ratio_original"].replace(0, np.nan)
        )
        .groupby(["dataset", "split"], dropna=False)
        .agg(
            n_crops=("sample_id", "count"),
            n_patients=("patient_id", "nunique"),
            bbox_w_mean=("bbox_w", "mean"),
            bbox_h_mean=("bbox_h", "mean"),
            crop_size_native_mean=("crop_size_native", "mean"),
            lesion_ratio_original_mean=("lesion_ratio_original", "mean"),
            lesion_ratio_crop_mean=("lesion_ratio_crop", "mean"),
            enrichment_mean=("enrichment", "mean"),
            enrichment_median=("enrichment", "median"),
            border_touch_rate=("crop_touches_border", "mean"),
        )
        .reset_index()
    )
    summary.to_csv(out_root / "roi_crop_summary.csv", index=False)

    print("[6/7] Validating all NPZ outputs")
    npz_paths = sorted([out_root / p for p in manifest_df["npz_path"].tolist()])
    validation_rows = [validate_npz_file(p, out_root, args.resize_size) for p in tqdm(npz_paths, desc="validate", unit="npz")]
    validation_df = pd.DataFrame(validation_rows)
    validation_df.to_csv(out_root / "npz_full_validation.csv", index=False)

    print("[7/7] Creating contact sheets and dataset card")
    # Global contact sheet and one per split.
    create_contact_sheet(sorted(overlay_paths), out_root / "contact_sheet.png")
    for split in ["train", "validation", "test", "external_inbreast"]:
        split_overlays = sorted((out_root / split).glob("*_overlay.png"))
        create_contact_sheet(split_overlays, out_root / f"contact_sheet_{split}.png")
    write_dataset_card(out_root, config)

    # Final verdict.
    split_counts = manifest_df.groupby(["dataset", "split"]).size().reset_index(name="n")
    npz_ok_rate = float(validation_df["ok"].mean()) if not validation_df.empty else 0.0
    cbis_test_present = bool(((manifest_df["dataset"] == "CBIS-DDSM") & (manifest_df["split"] == "test")).any())
    leakage_ok = True
    try:
        # Re-check on generated manifest at dataset+patient granularity.
        leaks = manifest_df.groupby(["dataset", "patient_id"])["split"].nunique()
        leakage_ok = bool((leaks <= 1).all())
    except Exception:
        leakage_ok = False

    enrichment = manifest_df["lesion_ratio_crop"] / manifest_df["lesion_ratio_original"].replace(0, np.nan)
    enrichment_mean = float(enrichment.mean())
    enrichment_median = float(enrichment.median())

    print("\n================ ROI CROPS 256 — FINAL SUMMARY ================")
    print(split_counts.to_string(index=False))
    print("---------------------------------------------------------------")
    print(f"Generated crops: {len(manifest_df)}")
    print(f"Failures logged: {len(failures)} -> {out_root / 'roi_crop_failures.csv'}")
    print(f"Patient leakage check: {'OK (0 fuite)' if leakage_ok else 'FAILED'}")
    print(f"NPZ validation: {validation_df['ok'].sum()}/{len(validation_df)} OK ({npz_ok_rate:.1%})")
    print(f"CBIS test bucket present: {'YES' if cbis_test_present else 'NO'}")
    print(f"Lesion enrichment mean:   x{enrichment_mean:.2f}")
    print(f"Lesion enrichment median: x{enrichment_median:.2f}")

    verdict_ok = leakage_ok and npz_ok_rate == 1.0 and cbis_test_present
    print("---------------------------------------------------------------")
    print(f"VERDICT: {'PASS' if verdict_ok else 'FAIL'}")
    print(f"Output root: {out_root.resolve()}")
    print("================================================================\n")

    if not verdict_ok:
        raise RuntimeError("Dataset generation completed but final verdict is FAIL. Inspect QC files.")


def parse_args(argv: Optional[Sequence[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate 256x256 ROI crops from full-field 448x448 CBIS-DDSM + INbreast package."
    )
    parser.add_argument("--source-zip", required=True, help="Path to CBIS-DDSM+InBreast-448.zip")
    parser.add_argument("--out-dir", default="ROI_Crops_256_v1", help="Output dataset root directory")
    parser.add_argument("--resize-size", type=int, default=DEFAULT_RESIZE_SIZE, help="Final ROI crop size, default 256")
    parser.add_argument("--margin-ratio", type=float, default=0.40, help="Bounding-box margin ratio, default 0.40")
    parser.add_argument("--seed", type=int, default=42, help="Random seed for patient-level split")
    parser.add_argument("--val-frac", type=float, default=0.15, help="CBIS validation fraction at patient level")
    parser.add_argument("--test-frac", type=float, default=0.15, help="CBIS test fraction at patient level")
    parser.add_argument("--overwrite", action="store_true", help="Delete out-dir if it already exists")
    args = parser.parse_args(argv)

    if args.resize_size <= 0:
        parser.error("--resize-size must be positive")
    if not (0.0 <= args.margin_ratio <= 2.0):
        parser.error("--margin-ratio should be in [0, 2]")
    if not (0.0 < args.val_frac < 1.0):
        parser.error("--val-frac must be in (0,1)")
    if not (0.0 < args.test_frac < 1.0):
        parser.error("--test-frac must be in (0,1)")
    if args.val_frac + args.test_frac >= 1.0:
        parser.error("--val-frac + --test-frac must be < 1")
    return args


if __name__ == "__main__":
    generate_dataset(parse_args())
'''

SCRIPT_PATH.write_text(SCRIPT_CODE, encoding='utf-8')
print('Script écrit :', SCRIPT_PATH)
print('Taille :', SCRIPT_PATH.stat().st_size, 'octets')

In [ ]:
import sys
import subprocess

cmd = [sys.executable, '-m', 'py_compile', str(SCRIPT_PATH)]
print('Commande :', ' '.join(cmd))
res = subprocess.run(cmd, text=True, capture_output=True)
print('STDOUT:', res.stdout)
print('STDERR:', res.stderr)
if res.returncode != 0:
    raise RuntimeError('Échec de compilation Python du script.')
print('Compilation Python : OK')

In [ ]:
import sys
import subprocess
from pathlib import Path

cmd = [
    sys.executable, str(SCRIPT_PATH),
    '--source-zip', str(SOURCE_ZIP),
    '--out-dir', str(OUT_DIR),
    '--resize-size', str(RESIZE_SIZE),
    '--margin-ratio', str(MARGIN_RATIO),
    '--seed', str(SEED),
    '--val-frac', str(VAL_FRAC),
    '--test-frac', str(TEST_FRAC),
]
if OVERWRITE:
    cmd.append('--overwrite')

print('Commande lancée :')
print(' '.join(cmd))

res = subprocess.run(cmd, text=True)
if res.returncode != 0:
    raise RuntimeError(f'La génération a échoué avec le code {res.returncode}. Inspectez roi_crop_failures.csv si créé.')
print('Génération terminée avec succès.')

In [ ]:
from pathlib import Path
import pandas as pd

out = Path(OUT_DIR)
expected_files = [
    'roi_crop_manifest.csv',
    'roi_crop_summary.csv',
    'npz_full_validation.csv',
    'roi_crop_failures.csv',
    'roi_crop_config.json',
    'ROI_CROPS_DATASET_CARD.md',
    'contact_sheet.png',
]
print('Dossier de sortie :', out)
for f in expected_files:
    p = out / f
    print(f'{f:35s}', 'OK' if p.exists() else 'MANQUANT')

manifest = pd.read_csv(out / 'roi_crop_manifest.csv')
summary = pd.read_csv(out / 'roi_crop_summary.csv')
validation = pd.read_csv(out / 'npz_full_validation.csv')
failures = pd.read_csv(out / 'roi_crop_failures.csv')

print('\nRésumé par dataset/split :')
display(summary)

print('\nValidation NPZ :')
print(validation['ok'].value_counts(dropna=False))

print('\nNombre de failures/orphelins journalisés :', len(failures))
if len(failures):
    display(failures.head(20))

leaks = manifest.groupby(['dataset', 'patient_id'])['split'].nunique().reset_index(name='n_splits')
leaks = leaks[leaks['n_splits'] > 1]
print('\nFuites patient détectées :', len(leaks))
if len(leaks):
    display(leaks.head(20))
    raise RuntimeError('Fuite patient détectée dans le manifeste final.')

cbis_test_present = ((manifest['dataset'] == 'CBIS-DDSM') & (manifest['split'] == 'test')).any()
print('Bucket CBIS test présent :', cbis_test_present)
if not cbis_test_present:
    raise RuntimeError('Bucket CBIS test absent.')

ok_rate = validation['ok'].mean()
print(f'Taux NPZ valides : {ok_rate:.1%}')
if ok_rate != 1.0:
    raise RuntimeError('Certains NPZ sont invalides. Inspectez npz_full_validation.csv.')


In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPyImage

contact = Path(OUT_DIR) / 'contact_sheet.png'
if contact.exists():
    display(IPyImage(filename=str(contact)))
else:
    print('contact_sheet.png non trouvée.')

In [ ]:
from pathlib import Path
import shutil
import hashlib

out = Path(OUT_DIR)
zip_path = Path(FINAL_ZIP)
if zip_path.exists():
    zip_path.unlink()

archive_base = str(zip_path).replace('.zip', '')
shutil.make_archive(archive_base, 'zip', root_dir=out.parent, base_dir=out.name)

sha = sha256_file(zip_path)
print('ZIP final :', zip_path)
print('Taille :', zip_path.stat().st_size, 'octets')
print('SHA256 :', sha)

pd.DataFrame([{
    'final_zip': str(zip_path),
    'sha256': sha,
    'size_bytes': zip_path.stat().st_size,
}]).to_csv(Path('/kaggle/working/ROI_Crops_256_v1_Kaggle_sha256.csv'), index=False)
print('SHA écrit : /kaggle/working/ROI_Crops_256_v1_Kaggle_sha256.csv')